In [ ]:
#Installing the libraries

In [ ]:
!pip install pypdf
!pip install sentence-transformers
!pip install faiss-cpu
!pip install google-generativeai

In [ ]:
#Uploading pdf

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
#Extracting Pdf text

In [ ]:
from pypdf import PdfReader

reader = PdfReader("sample.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print(text[:1000])

In [ ]:
#Chunk text
def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

chunks = chunk_text(text)

print("Chunks:", len(chunks))

In [ ]:
#Generating embeddings
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

print(embeddings.shape)

In [ ]:

print(embeddings.shape)

In [ ]:
print(context)

In [ ]:
#Creating FAISS index

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings)
)

print("FAISS index created")

In [ ]:
#Gemini configuration

In [ ]:
import google.generativeai as genai

genai.configure(
    api_key="GEMINI_API_KEY"
)

llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [ ]:
#Chatbot loop

In [ ]:
while True:

    query = input(
        "\nAsk Question (exit to stop): "
    )

    if query.lower() == "exit":
        break

    query_embedding = embedding_model.encode(
        [query]
    )

    D, I = index.search(
        np.array(query_embedding),
        k=3
    )

    retrieved_chunks = [
        chunks[idx]
        for idx in I[0]
    ]

    context = "\n".join(
        retrieved_chunks
    )

    prompt = f"""

    Answer only using the provided context.

    If the answer is not present,
    say:
    "I could not find the answer in the document."

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.generate_content(
        prompt
    )

    print("\nAnswer:")
    print(response.text)